# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd
data = pd.read_csv('data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  str    
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  str    
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  str    
 8   income_type       21525 non-null  str    
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  str    
dtypes: float64(2), int64(5), str(5)
memory usage: 2.0 MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
print(data.isna().sum())


children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64


**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
medians = {}
for income_type, group in data.groupby('income_type'):
    median_income = group['total_income'].median()
    medians[income_type] = median_income

print(data.isna().sum())




children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64


### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
print(data.groupby('income_type')['days_employed'].median().sort_values(ascending=False).round(2))

income_type
безработный        366413.65
пенсионер          365213.31
в декрете            3296.76
госслужащий          2689.37
сотрудник            1574.20
компаньон            1547.38
студент               578.75
предприниматель       520.85
Name: days_employed, dtype: float64


У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
print(data['children'].unique())

[ 1  0  3  2 -1  4 20  5]


**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data.drop(data[data['children'] == -1].index)
data = data.drop(data[data['children'] == 20].index)

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
print(data['children'].unique())

[1 0 3 2 4 5]


### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for income_type, median in medians.items():
    data.loc[data['income_type'] == income_type, 'days_employed'] = median

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
print(data.isna().sum())

children               0
days_employed          0
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2162
purpose                0
dtype: int64


### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = pd.to_numeric(data['total_income'], errors='coerce').fillna(0).astype(int)
data.info()
#Света, проверяю уже 2 раз, вроде бы все работает:)

<class 'pandas.DataFrame'>
Index: 21402 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21402 non-null  int64  
 1   days_employed     21402 non-null  float64
 2   dob_years         21402 non-null  int64  
 3   education         21402 non-null  str    
 4   education_id      21402 non-null  int64  
 5   family_status     21402 non-null  str    
 6   family_status_id  21402 non-null  int64  
 7   gender            21402 non-null  str    
 8   income_type       21402 non-null  str    
 9   debt              21402 non-null  int64  
 10  total_income      21402 non-null  int64  
 11  purpose           21402 non-null  str    
dtypes: float64(1), int64(6), str(5)
memory usage: 2.1 MB


### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].apply(str.lower)
data['family_status'] = data['family_status'].apply(str.lower)
data['gender'] = data['gender'].apply(str.lower)
data['income_type'] = data['income_type'].apply(str.lower)
data['purpose'] = data['purpose'].apply(str.lower)

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
print(data.duplicated().sum())

71


In [16]:
data = data.drop_duplicates().reset_index(drop=True)
print(data.duplicated().sum())

0


### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categor_zp(zp):
    if zp <= 30000:
        return 'E'
    elif zp > 30000 and zp <= 50000:
        return 'D'
    elif zp > 50000 and zp <= 200000:
        return 'C'
    elif zp > 200000 and zp <= 1000000:
        return 'B'
    else:
        return 'A'

In [18]:
data['total_income_category'] = data['total_income'].apply(categor_zp)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
print(data['purpose'].unique())

<StringArray>
[                         'покупка жилья',
                'приобретение автомобиля',
             'дополнительное образование',
                        'сыграть свадьбу',
                      'операции с жильем',
                            'образование',
                  'на проведение свадьбы',
                'покупка жилья для семьи',
                   'покупка недвижимости',
      'покупка коммерческой недвижимости',
             'покупка жилой недвижимости',
 'строительство собственной недвижимости',
                           'недвижимость',
             'строительство недвижимости',
     'на покупку подержанного автомобиля',
           'на покупку своего автомобиля',
  'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости',
                                  'жилье',
        'операции со своей недвижимостью',
                             'автомобили',
                  'заняться образованием',
       'сделка с подержанным автомобилем

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [22]:
children_category_pivot = data.pivot_table(index=['children'], values = 'debt', aggfunc = {'count', 'sum'})

children_category_pivot = children_category_pivot.rename(columns = {'count':'total', 'sum':'debt'})

children_category_pivot['share'] = children_category_pivot['debt']/children_category_pivot['total']

children_category_pivot = children_category_pivot.sort_values(by = ['share'], ascending=False)

children_category_pivot['share'] = children_category_pivot['share'].apply(lambda x: '{:.2%}'.format(x))

children_category_pivot


,total,debt,share
children,,,
4,41,4,9.76%
2,2052,194,9.45%
1,4808,444,9.23%
3,330,27,8.18%
0,14091,1063,7.54%
5,9,0,0.00%


**Вывод:** Можно сделать вывод, что семьи с 1, 2, или 4 детьми наиболее подвержены просрочкам по кредитам, в среднем это 9.5%. 3 детей у нас в середине - это 8.2%. Без детей - наиболее ответственные заемщики, просрочка всего 7.54%. По категории 5 детей, вывод сделать не возможно, так как не хватает данных.
Общий вывод не сделать однозначно, как и почему статистика получилось именно такой. С одной стороны родители с детьми должны быть более отвественны, нежели люди без детей, с другой стороны возможно обуславливается тем, что на детей нужны дополнительные расходы и возможно в таком случае возникает финансовая нестабильность.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [23]:
family_category_pivot = data.pivot_table(index=['family_status'], values = 'debt', aggfunc = {'count', 'sum'})

family_category_pivot = family_category_pivot.rename(columns = {'count':'total', 'sum':'debt'})

family_category_pivot['share'] = family_category_pivot['debt']/family_category_pivot['total']

family_category_pivot = family_category_pivot.sort_values(by = 'share', ascending=False)

family_category_pivot['share'] = family_category_pivot['share'].apply(lambda x: '{:.2%}'.format(x))

family_category_pivot

,total,debt,share
family_status,,,
не женат / не замужем,2796,273,9.76%
гражданский брак,4134,385,9.31%
женат / замужем,12261,927,7.56%
в разводе,1189,84,7.06%
вдовец / вдова,951,63,6.62%


**Вывод:** Можно сделать следующий вывод: "вдовец/вдова" - наименьшее количество просроченных кредитов, наибольшее в категории - "гражданский брак" и "Не женат / не замужем".

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [24]:
debt_from_zp = pd.pivot_table(data, index=['total_income_category'], values='debt', aggfunc={'sum', 'count'}, fill_value=0)

debt_from_zp = debt_from_zp.rename(columns={'count':'total', 'sum':'debt'})

debt_from_zp['share'] = debt_from_zp['debt'] / debt_from_zp['total']

debt_from_zp['share'] = debt_from_zp['share'].map('{:,.2%}'.format)

debt_from_zp = debt_from_zp.sort_values(by = 'share', ascending = False)

debt_from_zp

,total,debt,share
total_income_category,,,
C,13831,1183,8.55%
E,2113,172,8.14%
A,25,2,8.00%
B,5013,354,7.06%
D,349,21,6.02%


**Вывод:** мало данных по уровню зарплат, конкретный вывод сделать не представляется возможным.

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [25]:
debt_from_category = pd.pivot_table(data, index=['purpose_category'], values='debt', aggfunc={'sum', 'count'}, fill_value=0)

debt_from_category['share'] = debt_from_category['sum'] / debt_from_category['count']

debt_from_category['share'] = debt_from_category['share'].map('{:,.2%}'.format)

debt_from_category = debt_from_category.sort_values('share', ascending=False)

debt_from_category

,count,sum,share
purpose_category,,,
операции с автомобилем,4279,400,9.35%
получение образования,3988,369,9.25%
проведение свадьбы,2313,183,7.91%
операции с недвижимостью,10751,780,7.26%


**Вывод:** заемщики берущие кредиты на недвижимость - наиболее отвественны. Авто кредит, кредит на обучение - примерно одинаково допускается просрочка.

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* технические проблемы

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* Заполнение пропусков медианным значением — полезный метод обработки пропущенных данных для количественных переменных, так как медиана сохраняет центральную тенденцию, менее чувствительна к выбросам и экстремальным значениям, проста в интерпретации, устойчива к распределению и предотвращает смещение среднего.

### Шаг 4: общий вывод.

Отвечая на поставленный вопрос "Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок", могу дать следующий ответ - семеное положение и количество детей влияет на факт погашения кредита в строк:
-заемщики с официально оформленными отношениями (или которые в прошлом были в официальном в браке) и не имеющие детей - самые ответственные заемщики;
-заемщики, состоящие в неофициальном браке или находящиеся без отношений, при этом имеющие 1 или 2 детей - самые менее ответственные заемщики.